# _**Matsch_P2_Ref_007: fidelity run against GEOtop 3.0**_

    -Authors: GEO2GEO project (Waterjade for Rigonian Lab)
    -License: GPL v3, as WHETGEO-1D
-----
The snow+soil column (`SnowSoilSolver1DMain`, configuration *fidelity*) against GEOtop 3.0 (commit 1f1fc5af) on the official 1D test **Matsch_P2_Ref_007 without canopy and corrected** (`casi/Matsch_P2_Ref_007/run.sh corretta`): `CanopyFraction = 0`, and three errors of the official case removed. The three errors:

- the meteo file's `JDfrom0` column was one day behind its dates, and GEOtop trusts it (`fixing_dates`, `meteodata.cc:583`). The column is dropped and the dates come from `Date`. This needs the inline `!` comments stripped from `geotop.inpts`: GEOtop's parser keeps them, so `MeteoStationStandardTime = 1 ! added by ... 2.1` was read as 1e21;
- `ActualOrProjectedArea = 1`, without which GEOtop projects the precipitation twice in a point simulation (`energy.balance.cc:299-304`);
- an empty `CloudTrans` column, which stops GEOtop from computing the daily cloudiness from the measured shortwave (`input.cc:405-408`).

This reference no longer equals the official test output. Station P2 in Val Mazia, 1549 m, slope 21.4°, 2 October to 2 November 2009, steps of 900 s, hourly outputs.

**What the comparison tests.** The forcing is GEOtop's own. The temporary tools of `core.snowsoil.fidelity` compute it from GEOtop's input files and reproduce the point output of the corrected run to the printed digits (`GeotopForcingTest`); every quantity agrees within 5e-7, including the averaged cloudiness. Two things come from GEOtop's *output* instead (the fallback of DN-14), because the column does not compute them:

- the **bare-soil surface temperature** `Tsurface`, imposed as a Dirichlet top when there is no snow (the column has no bare-soil energy balance);
- the **soil evaporation** `Evap_surface`, a sink of the top cell without snow (evapotranspiration comes from GEOET in GEOframe).

So this is a **conditioned comparison** (VAL-1): without snow it tests the soil (heat, Richards, freezing) under GEOtop's surface temperature; with snow it tests the snowpack and its surface balance.

**Tolerances.** VAL-1 has no tolerance table yet. The thresholds below are a **proposal**, written before reading the results of this run, to be discussed for the table; they are not approved.

Run `cases/snowsoil/matsch_p2_ref_007/run.sh` first: it writes the forcing and the outputs read here.

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from WHETGEO1D_SnowSoilOutputs import *
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({"axes.edgecolor": "0.8", "axes.linewidth": 0.6, "grid.linewidth": 0.3, "font.family": "serif",
                     "axes.labelsize": 13, "font.size": 13, "legend.fontsize": 10})
pd.set_option('display.float_format', '{:.4g}'.format)

## Define input

In [ ]:
case = os.environ.get('MATSCH_CASE', os.path.join('..', 'cases', 'snowsoil', 'matsch_p2_ref_007'))
ours_path = os.path.join(case, 'output')
geotop_path = os.path.join(case, 'geotop', 'output-tabs')

## Read both runs
The CSV files of the column carry GEOtop's names and units, and dates in GEOtop's local standard time (`outputStandardTime = 1`), so the columns align by name and the rows by date.

In [ ]:
ours, header = read_csv(os.path.join(ours_path, 'point.csv'))
gt, _ = read_csv(os.path.join(geotop_path, 'point0001.txt'))
idx = ours.index.intersection(gt.index)
ours, gt = ours.loc[idx], gt.loc[idx]
print(len(idx), 'hours,', idx[0], '-', idx[-1])
for k in ['code_version', 'configuration', 'geotopSnowParameters', 'bareSoilTop', 'bottomBC', 'soilWaterBottomBC']:
    print(f'{k:22s} {header.get(k)}')
profiles = {}
for name, geotop_name in [('soilTz', 'soilTz0001'), ('thetaliq', 'thetaliq0001'), ('thetaice', 'thetaice0001'), ('psiz', 'psiz0001')]:
    a, _ = read_csv(os.path.join(ours_path, name + '.csv'))
    b, _ = read_csv(os.path.join(geotop_path, geotop_name + '.txt'))
    profiles[name] = (a.loc[idx, profile_columns(a)], b.loc[idx, profile_columns(b)])
depths = profile_columns(profiles['soilTz'][0])
print('soil cells [mm]:', depths)

## Proposed tolerances (VAL-1, not approved)

In [ ]:
tolerances = {
    'snow depth RMSE [mm]': 2.0,
    'snow depth max |diff| [mm]': 5.0,
    'SWE RMSE [mm]': 0.2,
    'SWE peak, relative': 0.10,
    'melt total, relative': 0.10,
    'snow runoff total, relative': 0.10,
    'Tsurface with snow RMSE [K]': 0.5,
    'surface fluxes with snow RMSE [W/m2]': 5.0,
    'soil T bias [K]': 0.1,
    'soil T RMSE [K]': 0.2,
    'theta_liq RMSE [-]': 0.01,
    'theta_ice max, top cell [-]': 0.02,
    'frost depth max [cells]': 0,
}
pd.Series(tolerances)

## Snow

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
for a, col, label in [(ax[0], 'snow_depth[mm]', 'HS [mm]'), (ax[1], 'snow_water_equivalent[mm]', 'SWE [mm]'),
                      (ax[2], 'Tsurface[C]', 'T$_s$ [°C]')]:
    a.plot(gt.index.to_numpy(), gt[col].to_numpy(), 'k', lw=1.2, label='GEOtop 3.0')
    a.plot(ours.index.to_numpy(), ours[col].to_numpy(), 'C0', lw=1.2, label='port, fidelity')
    a.set_ylabel(label)
    a.grid(color='grey', linestyle='-', linewidth=0.3)
ax[0].legend()
ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.tight_layout()

The three snow events, closer: 11 October (a trace), 21 October and 1–2 November.

In [ ]:
events = [('2009-10-11 08:00', '2009-10-11 16:00'), ('2009-10-21 00:00', '2009-10-22 08:00'), ('2009-11-01 10:00', '2009-11-02 14:00')]
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
for j, (a, b) in enumerate(events):
    o, g = ours.loc[a:b], gt.loc[a:b]
    for i, col in enumerate(['snow_water_equivalent[mm]', 'snow_depth[mm]']):
        ax[i, j].plot(g.index.to_numpy(), g[col].to_numpy(), 'k', label='GEOtop')
        ax[i, j].plot(o.index.to_numpy(), o[col].to_numpy(), 'C0', label='port')
        ax[i, j].set_ylabel(col)
        ax[i, j].xaxis.set_major_formatter(mdates.DateFormatter('%d %Hh'))
        ax[i, j].grid(color='grey', linestyle='-', linewidth=0.3)
ax[0, 0].legend()
plt.tight_layout()

Snow runoff: the column writes the outflow from the base of the pack (`snowRunoff`); GEOtop's point file has none. GEOtop's `Prain_rain_on_snow` is not the rain entering the pack (0.5 mm against the 3–6 mm that fall on snow in either model), so both runoffs are closed here the same way from hourly values: snowfall + rain falling while the pack of the previous hour exists − sublimation − ΔSWE (sublimation positive in both, checked on the totals). The column's own `snowRunoff` is listed too.

In [ ]:
def rain_on_snow(d):
    return d['Prain_under_canopy[mm]'][d['snow_water_equivalent[mm]'].shift(1, fill_value=0.0) > 0].sum()

def closed_runoff(d):
    dswe = d['snow_water_equivalent[mm]'].iloc[-1] - d['snow_water_equivalent[mm]'].iloc[0]
    return d['Psnow_under_canopy[mm]'].sum() + rain_on_snow(d) - d['snow_subl[mm]'].sum() - dswe

runoff_gt, runoff_ours = closed_runoff(gt), closed_runoff(ours)
snow_table = pd.DataFrame({
    'GEOtop': [gt['snow_water_equivalent[mm]'].max(), gt['snow_depth[mm]'].max(), gt['snow_melted[mm]'].sum(), gt['snow_subl[mm]'].sum(),
               rain_on_snow(gt), runoff_gt, np.nan, (gt['snow_water_equivalent[mm]'] > 0).sum()],
    'port': [ours['snow_water_equivalent[mm]'].max(), ours['snow_depth[mm]'].max(), ours['snow_melted[mm]'].sum(), ours['snow_subl[mm]'].sum(),
             rain_on_snow(ours), runoff_ours, ours['snowRunoff[mm]'].sum(), (ours['snow_water_equivalent[mm]'] > 0).sum()]},
    index=['SWE max [mm]', 'HS max [mm]', 'melt [mm]', 'sublimation [mm]', 'rain on the pack [mm]', 'snow runoff, closed [mm]',
           'snow runoff, written [mm]', 'hours with snow'])
snow_table

## Surface fluxes with snow in both models
Without snow the column's surface is GEOtop's `Tsurface` and the column computes no balance (NaN), so the fluxes are compared only in the hours where both have snow.

In [ ]:
both = (ours['snow_water_equivalent[mm]'] > 0) & (gt['snow_water_equivalent[mm]'] > 0)
flux_cols = ['SWnet[W/m2]', 'LWnet[W/m2]', 'H[W/m2]', 'LE[W/m2]', 'Tsurface[C]']
rows = {}
for col in flux_cols:
    d = (ours.loc[both, col] - gt.loc[both, col])
    rows[col] = {'hours': int(both.sum()), 'mean GEOtop': gt.loc[both, col].mean(), 'mean port': ours.loc[both, col].mean(),
                 'bias': d.mean(), 'RMSE': np.sqrt((d ** 2).mean())}
flux_table = pd.DataFrame(rows).T
flux_table

## Soil temperature

In [ ]:
T_ours, T_gt = profiles['soilTz']
rows = {}
for d in depths:
    diff = T_ours[d] - T_gt[d]
    rows[d] = {'bias [K]': diff.mean(), 'RMSE [K]': np.sqrt((diff ** 2).mean()), 'max |diff| [K]': diff.abs().max(),
               'min GEOtop [C]': T_gt[d].min(), 'min port [C]': T_ours[d].min()}
soilT_table = pd.DataFrame(rows).T
soilT_table.index.name = 'depth [mm]'
soilT_table

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for d, color in zip([5.0, 25.0, 85.0, 275.0, 650.0], ['C0', 'C1', 'C2', 'C3', 'C4']):
    ax[0].plot(T_gt.index.to_numpy(), T_gt[d].to_numpy(), color=color, lw=1.6, alpha=0.5)
    ax[0].plot(T_ours.index.to_numpy(), T_ours[d].to_numpy(), color=color, lw=0.9, ls='--', label=f'{d:.0f} mm')
    ax[1].plot(T_ours.index.to_numpy(), (T_ours[d] - T_gt[d]).to_numpy(), color=color, lw=0.9, label=f'{d:.0f} mm')
ax[0].set_ylabel('T [°C] (GEOtop solid, port dashed)')
ax[1].set_ylabel('port − GEOtop [K]')
ax[0].legend(ncol=5)
for a in ax:
    a.grid(color='grey', linestyle='-', linewidth=0.3)
ax[1].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.tight_layout()

## Soil water and frozen soil
Liquid and ice content at the top cells, and the frost depth: the depth of the centre of the deepest cell with θ$_i$ > 10⁻³ (on the grid: cells of 10 mm at the top).

In [ ]:
L_ours, L_gt = profiles['thetaliq']
I_ours, I_gt = profiles['thetaice']
rows = {}
for d in depths[:6]:
    rows[d] = {'theta_liq RMSE': np.sqrt(((L_ours[d] - L_gt[d]) ** 2).mean()), 'theta_liq bias': (L_ours[d] - L_gt[d]).mean(),
               'theta_ice max GEOtop': I_gt[d].max(), 'theta_ice max port': I_ours[d].max()}
water_table = pd.DataFrame(rows).T
water_table.index.name = 'depth [mm]'

def frost_depth(ice):
    cols = np.array(profile_columns(ice))
    frozen = ice[cols].values > 1e-3
    return pd.Series(np.where(frozen.any(axis=1), [cols[r].max() if r.any() else 0 for r in frozen], 0.0), index=ice.index)

fd_ours, fd_gt = frost_depth(I_ours), frost_depth(I_gt)
frost_table = pd.Series({'max frost depth GEOtop [mm]': fd_gt.max(), 'max frost depth port [mm]': fd_ours.max(),
                         'hours frozen GEOtop': int((fd_gt > 0).sum()), 'hours frozen port': int((fd_ours > 0).sum()),
                         'hours with a different frost cell': int((fd_gt != fd_ours).sum())})
display(water_table)
frost_table

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
ax[0].plot(L_gt.index.to_numpy(), L_gt[5.0].to_numpy(), 'k', label='GEOtop')
ax[0].plot(L_ours.index.to_numpy(), L_ours[5.0].to_numpy(), 'C0', label='port')
ax[0].set_ylabel('$\\theta_w$ at 5 mm')
ax[1].plot(I_gt.index.to_numpy(), I_gt[5.0].to_numpy(), 'k')
ax[1].plot(I_ours.index.to_numpy(), I_ours[5.0].to_numpy(), 'C0')
ax[1].set_ylabel('$\\theta_i$ at 5 mm')
ax[2].step(fd_gt.index.to_numpy(), fd_gt.to_numpy(), 'k', where='pre')
ax[2].step(fd_ours.index.to_numpy(), fd_ours.to_numpy(), 'C0', where='pre')
ax[2].set_ylabel('frost depth [mm]')
ax[2].invert_yaxis()
ax[0].legend()
for a in ax:
    a.grid(color='grey', linestyle='-', linewidth=0.3)
ax[2].xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
plt.tight_layout()

## Against the proposed tolerances

In [ ]:
def rmse(a, b):
    return float(np.sqrt(((a - b) ** 2).mean()))

hs_o, hs_g = ours['snow_depth[mm]'], gt['snow_depth[mm]']
swe_o, swe_g = ours['snow_water_equivalent[mm]'], gt['snow_water_equivalent[mm]']
checks = [
    ('snow depth RMSE [mm]', rmse(hs_o, hs_g), tolerances['snow depth RMSE [mm]']),
    ('snow depth max |diff| [mm]', float((hs_o - hs_g).abs().max()), tolerances['snow depth max |diff| [mm]']),
    ('SWE RMSE [mm]', rmse(swe_o, swe_g), tolerances['SWE RMSE [mm]']),
    ('SWE peak, relative', abs(swe_o.max() / swe_g.max() - 1), tolerances['SWE peak, relative']),
    ('melt total, relative', abs(ours['snow_melted[mm]'].sum() / gt['snow_melted[mm]'].sum() - 1), tolerances['melt total, relative']),
    ('snow runoff total, relative', abs(runoff_ours / runoff_gt - 1), tolerances['snow runoff total, relative']),
    ('Tsurface with snow RMSE [K]', flux_table.loc['Tsurface[C]', 'RMSE'], tolerances['Tsurface with snow RMSE [K]']),
]
for col in ['SWnet[W/m2]', 'LWnet[W/m2]', 'H[W/m2]', 'LE[W/m2]']:
    checks.append((f'{col} with snow RMSE', flux_table.loc[col, 'RMSE'], tolerances['surface fluxes with snow RMSE [W/m2]']))
for d in [5.0, 15.0, 25.0, 85.0, 175.0, 450.0, 925.0]:
    checks.append((f'soil T {d:.0f} mm |bias| [K]', abs(soilT_table.loc[d, 'bias [K]']), tolerances['soil T bias [K]']))
    checks.append((f'soil T {d:.0f} mm RMSE [K]', soilT_table.loc[d, 'RMSE [K]'], tolerances['soil T RMSE [K]']))
for d in depths[:3]:
    checks.append((f'theta_liq {d:.0f} mm RMSE', water_table.loc[d, 'theta_liq RMSE'], tolerances['theta_liq RMSE [-]']))
checks.append(('theta_ice max 5 mm |diff|', abs(I_ours[5.0].max() - I_gt[5.0].max()), tolerances['theta_ice max, top cell [-]']))
cell = {d: i for i, d in enumerate(depths)}
checks.append(('frost depth max [cells]', abs(cell.get(fd_ours.max(), -1) - cell.get(fd_gt.max(), -1)), tolerances['frost depth max [cells]']))
summary = pd.DataFrame(checks, columns=['metric', 'value', 'tolerance']).set_index('metric')
summary['within'] = summary['value'] <= summary['tolerance']
summary

## Reading the differences

The forcing is not among the causes: the tools reproduce GEOtop's own forcing of the corrected run to the printed digits (`GeotopForcingTest`). The three snow events (12 October, a trace; 22 October; 2 November) are thin packs, never above 1 mm of SWE. The differences beyond the proposed tolerances, with their attribution:

1. **HS of a melting pack (GEOtop behaviour).** On 2 November GEOtop's single layer keeps its thickness, 23.2 mm, while its SWE melts from 1.04 to 0.01 mm: the density falls to below 1 kg m⁻³. On 22 October GEOtop likewise keeps 2.7 mm of depth with 0.001 mm of SWE for a day. The port shrinks the cell with the melt. This accounts for the HS max difference of 20 mm and is not physics to reproduce.
2. **Thin snow (porting, not yet ported).** GEOtop changes regime for thin packs, and the port does not yet (DN-14):
   - below `ThresSnowSoilRough` = 100 mm GEOtop uses the soil roughness, 100 mm against 1.6 mm;
   - below 10 mm it uses the soil emissivity.

   With snow the port's surface is 0.3 K warmer on average, LE is 5 W m⁻² higher (sublimation 0.08 against 0.04 mm) and H differs by 7 W m⁻² RMSE.
3. **Melt and runoff (follows from 2).** Melt is 11 % lower and runoff 21 % lower in the port. The port's pack exists for fewer hours with rain on it (rain on the pack 4.7 against 6.5 mm), and those hours make up most of the runoff.
4. **Soil near the surface (conditioned top).** The bare-soil top is GEOtop's *hourly mean* `Tsurface`, held over the four 900 s steps of each hour, while GEOtop moves it every step. It is imposed at the surface face of the top cell, while GEOtop's is the temperature of its top layer. Both act within the day in the top 10 cm: RMSE 0.18–0.41 K, bias ≤ 0.01 K. Below 10 cm the soil is within tolerance; the deep bias of +0.13 K seen against the uncorrected reference has gone (now +0.04 K at 0.9 m). In the top cell the port freezes more, θ$_i$ 0.43 against 0.38 (to isolate: the Dirichlet face, evaporation applied as a sink of that cell, the freezing curve). The frost reaches the same cell (15 mm); 4 of 767 hours differ.
5. **Slope in Richards (configuration).** GEOtop's slope in the gravity term of Richards is not mapped: the column is vertical, with an impervious bottom and the water table below it.

**Nothing has been tuned** to bring the port closer to GEOtop.